# FICOS FLEX number discrepancy reconciliation

Fresh audit of the competing historical-FLEX figures: `-$4,228,965`, `-$254,155`, and approximately `-$25K`. This notebook does not trust previous reports as evidence. It reconstructs populations, formulas, signs, and aggregations from the canonical dataset and executable source logic.

In [ ]:
import os, sys, subprocess, hashlib, re
from pathlib import Path
REPO = Path('/content/FICOS-Platform')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/SSOHEB/FICOS-Platform.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pandas','numpy','scikit-learn','pyyaml'], check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
DATA=REPO/'data'/'modeling_dataset.csv'; df=pd.read_csv(DATA); df['date']=pd.to_datetime(df['date']); df=df.sort_values('date').reset_index(drop=True)
print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()); print('dataset_sha256:',hashlib.sha256(DATA.read_bytes()).hexdigest())

In [ ]:
VESSELS=['panamax','supramax','handy','cape']; FEATURES=[c for c in df.columns if c!='date' and not c.startswith('target_') and not c.startswith('dir_')]
FOLDS=[
 {'year':2021,'train_end':'2019-12-24','val_start':'2020-01-03','val_end':'2020-12-24','test_start':'2021-01-05','test_end':'2021-12-31'},
 {'year':2022,'train_end':'2020-12-24','val_start':'2021-01-05','val_end':'2021-12-24','test_start':'2022-01-03','test_end':'2022-12-30'},
 {'year':2023,'train_end':'2021-12-24','val_start':'2022-01-03','val_end':'2022-12-23','test_start':'2023-01-03','test_end':'2023-12-29'},
 {'year':2024,'train_end':'2022-12-23','val_start':'2023-01-03','val_end':'2023-12-22','test_start':'2024-01-02','test_end':'2024-12-31'},
 {'year':2025,'train_end':'2023-12-22','val_start':'2024-01-02','val_end':'2024-12-24','test_start':'2025-01-02','test_end':'2025-12-31'}]
VOYAGE=20.0; IDLE=2500.0; WAIT_DAYS=1.0; FLEX_DAYS=0.25; TREES=100; SEED=42
def model_fold(fold,vessel):
 target=f'target_{vessel}_1d'; valid=df[vessel].notna() & df[target].notna()
 tr=(df.date<=fold['train_end'])&valid; va=(df.date>=fold['val_start'])&(df.date<=fold['val_end'])&valid; te=(df.date>=fold['test_start'])&(df.date<=fold['test_end'])&valid
 def X(m): return np.nan_to_num(df.loc[m,FEATURES].to_numpy(),nan=0,posinf=0,neginf=0)
 def y(m): return df.loc[m,target].to_numpy()-df.loc[m,vessel].to_numpy()
 sc=StandardScaler().fit(X(tr)); sel=SelectKBest(f_regression,k=min(30,len(FEATURES))).fit(sc.transform(X(tr)),y(tr)); rf=RandomForestRegressor(n_estimators=TREES,max_depth=5,random_state=SEED,n_jobs=1).fit(sel.transform(sc.transform(X(tr))),y(tr))
 vp=rf.predict(sel.transform(sc.transform(X(va)))); tp=rf.predict(sel.transform(sc.transform(X(te))))
 return {'val_pred':vp,'val_base':df.loc[va,vessel].to_numpy(),'val_true':df.loc[va,target].to_numpy(),'pred':tp,'base':df.loc[te,vessel].to_numpy(),'true':df.loc[te,target].to_numpy(),'date':df.loc[te,'date'].dt.strftime('%Y-%m-%d').to_numpy()}

In [ ]:
canonical=[]; exp06=[]; threshold_rows=[]
for fold in FOLDS:
 for vessel in VESSELS:
  r=model_fold(fold,vessel); residual=r['val_true']-r['val_base']-r['val_pred']; p10=float(np.percentile(residual,10)); p90=float(np.percentile(residual,90))
  cand=np.linspace(-300,-25,50); valnet=[np.sum(r['val_base']*VOYAGE-np.where(r['val_pred']<t,r['val_true']*VOYAGE+IDLE,r['val_base']*VOYAGE)) for t in cand]; tau=float(cand[int(np.argmax(valnet))])
  threshold_rows.append({'year':fold['year'],'vessel':vessel,'threshold':tau,'train_end':fold['train_end'],'val_end':fold['val_end'],'test_start':fold['test_start']})
  for i in range(len(r['true'])):
   base,true,pred=r['base'][i],r['true'][i],r['pred'][i]; spot=base*VOYAGE; wait=true*VOYAGE+IDLE*WAIT_DAYS; flex=(base+0.5*(true-base))*VOYAGE+IDLE*FLEX_DAYS
   cdec='NOW' if (pred>p90 and pred/(base+1e-8)>0.01) else ('WAIT' if (pred<p10 and pred/(base+1e-8)<-0.01) else 'FLEXIBLE')
   edec='WAIT' if pred<tau else 'SPOT_INDEX'
   common={'date':r['date'][i],'year':fold['year'],'vessel':vessel,'base_rate':base,'true_rate':true,'realized_delta':true-base,'pred_delta':pred,'spot_cost':spot,'wait_cost':wait,'historical_flex_cost':flex}
   canonical.append({**common,'decision':cdec,'historical_flex_delta':flex-spot,'policy_cost':spot if cdec=='NOW' else (wait if cdec=='WAIT' else flex),'formulation':'canonical_p10_p90'})
   exp06.append({**common,'decision':edec,'historical_flex_delta':flex-spot,'current_policy_cost':spot if edec!='WAIT' else wait,'mixed_historical_policy_cost':wait if edec=='WAIT' else flex,'formulation':'exp06_threshold'})
canonical=pd.DataFrame(canonical); exp06=pd.DataFrame(exp06); threshold_rows=pd.DataFrame(threshold_rows)
print('canonical population:',len(canonical)); print('EXP-06 population:',len(exp06)); display(threshold_rows)

In [ ]:
def report(name,x,mask,cost_col):
 y=x.loc[mask].copy(); y['delta']=y['spot_cost']-y[cost_col]
 return {'claim':name,'N_before':len(x),'N_after':len(y),'WAIT_N':int((y.decision=='WAIT').sum()),'nonWAIT_FLEX_N':int(y.decision.isin(['FLEXIBLE','SPOT_INDEX']).sum()),'years':f"{y.year.min()}-{y.year.max()}",'vessels':','.join(sorted(y.vessel.unique())),'formula':cost_col,'recomputed':float(y.delta.sum()),'baseline':'spot_cost','sign':'spot_cost - policy_cost'}
audit=pd.DataFrame([
 report('-$4,228,965',canonical,canonical.decision=='FLEXIBLE','policy_cost'),
 report('-$254,155',exp06,np.ones(len(exp06),dtype=bool),'mixed_historical_policy_cost'),
 report('EXP06 pure historical FLEX only',exp06,exp06.decision=='SPOT_INDEX','historical_flex_cost')])
display(audit)
print('Canonical historical FLEX formula: (base + 0.5 * (true-base)) * 20 + 2500 * 0.25')
print('The -$254,155 calculation is mixed: WAIT rows use WAIT cost; only non-WAIT rows use historical FLEX cost.')
print('The approximately -$25K claim: SOURCE NOT FOUND by text/source search in this checkout; no population or formula is asserted.')

In [ ]:
# Raw row-level evidence and source provenance.
OUT=REPO/'outputs'/'flex_discrepancy_colab'; OUT.mkdir(parents=True,exist_ok=True)
canonical.to_csv(OUT/'canonical_historical_flex_rows.csv',index=False); exp06.to_csv(OUT/'exp06_historical_flex_rows.csv',index=False); audit.to_csv(OUT/'three_number_reconciliation.csv',index=False); threshold_rows.to_csv(OUT/'walkforward_threshold_audit.csv',index=False)
provenance=pd.DataFrame([
 {'claim':'-$4,228,965','source':'tests/reproducibility/test_reproducibility_regression.py:191-196; scripts/forensic_audit_recomputation.py:113-118','population':'canonical decision == FLEXIBLE','status':'recomputed here'},
 {'claim':'-$254,155','source':'ml/notebooks/05_proof/FICOS_FINAL_FORENSIC_ECONOMIC_RECONCILIATION_COLAB.ipynb sensitivity cell','population':'all EXP-06 rows; WAIT uses WAIT cost, non-WAIT uses historical FLEX cost','status':'recomputed here'},
 {'claim':'approximately -$25K','source':'not found in repository text/source search','population':'UNKNOWN','status':'UNVERIFIED'}])
display(provenance); print('Evidence directory:',OUT)

In [ ]:
# Final single-run ledger and exports. Uses only the fresh dataframes above.
exp04 = exp06.copy(); exp04['decision'] = np.where(exp04['pred_delta'] < -125.0, 'WAIT', 'NON_WAIT')
exp04['current_contribution'] = np.where(exp04['decision'] == 'WAIT', exp04['spot_cost'] - exp04['wait_cost'], 0.0)
def metric_frame(x, decision_col, contribution_col, label):
    w = x[x[decision_col] == 'WAIT']; nw = x[x[decision_col] != 'WAIT']
    return {'experiment': label, 'N': len(x), 'WAIT_N': len(w), 'non_WAIT_N': len(nw), 'WAIT_precision': float((np.sign(w.true_rate-w.base_rate)==np.sign(w.pred_delta)).mean()) if len(w) else np.nan, 'gross_WAIT_gains': float(w.loc[w[contribution_col] > 0, contribution_col].sum()), 'gross_WAIT_losses': float(w.loc[w[contribution_col] < 0, contribution_col].sum()), 'net_WAIT_value': float(w[contribution_col].sum()), 'non_WAIT_contribution': float(nw[contribution_col].sum()), 'total_net_savings': float(x[contribution_col].sum()), 'baseline_spot_cost': float(x.spot_cost.sum())}
exp06['current_contribution'] = exp06.spot_cost - exp06.current_policy_cost
final_rows = [metric_frame(exp04, 'decision', 'current_contribution', 'EXP-04'), metric_frame(exp04[exp04.year == 2025], 'decision', 'current_contribution', 'EXP-04 2025'), metric_frame(exp06, 'decision', 'current_contribution', 'EXP-06'), metric_frame(exp06[exp06.year == 2025], 'decision', 'current_contribution', 'EXP-06 2025')]
final_reconciliation = pd.DataFrame(final_rows); display(final_reconciliation)
canonical_flex = canonical[canonical.decision == 'FLEXIBLE'].copy(); exp06_nonwait = exp06[exp06.decision == 'SPOT_INDEX'].copy(); exp06_wait = exp06[exp06.decision == 'WAIT'].copy()
flex_ledger = pd.DataFrame([{'claim':'-$4,228,965','population':'canonical FLEXIBLE','N':len(canonical_flex),'formula':'spot - historical_flex_cost','result':float((canonical_flex.spot_cost-canonical_flex.historical_flex_cost).sum()),'status':'RECOMPUTED'}, {'claim':'EXP-06 non-WAIT historical FLEX','population':'EXP-06 SPOT_INDEX','N':len(exp06_nonwait),'formula':'spot - historical_flex_cost','result':float((exp06_nonwait.spot_cost-exp06_nonwait.historical_flex_cost).sum()),'status':'RECOMPUTED'}, {'claim':'-$254,155','population':'EXP-06 all rows mixed','N':len(exp06),'formula':'WAIT uses WAIT cost; non-WAIT uses historical FLEX cost','result':float(exp06_wait.current_contribution.sum()+(exp06_nonwait.spot_cost-exp06_nonwait.historical_flex_cost).sum()),'status':'RECOMPUTED'}, {'claim':'approximately -$25K','population':'UNKNOWN','N':np.nan,'formula':'UNKNOWN','result':np.nan,'status':'NOT FOUND unless a source is printed above'}]); display(flex_ledger)
decomp = exp06_nonwait.assign(historical_flex_delta=exp06_nonwait.spot_cost-exp06_nonwait.historical_flex_cost).groupby(['year','vessel']).agg(N=('date','size'), mean_base_rate=('base_rate','mean'), mean_true_rate=('true_rate','mean'), mean_realized_delta=('realized_delta','mean'), mean_historical_flex_cost=('historical_flex_cost','mean'), mean_spot_cost=('spot_cost','mean'), mean_incremental_flex_delta=('historical_flex_delta','mean')).reset_index()
ledger = pd.DataFrame([{'experiment':'EXP-04','year_scope':'all','decision':'WAIT','N':int((exp04.decision=='WAIT').sum()),'formula':'spot - WAIT_cost','economic_contribution':float(exp04.current_contribution.sum())},{'experiment':'EXP-06','year_scope':'all','decision':'WAIT','N':len(exp06_wait),'formula':'spot - WAIT_cost','economic_contribution':float(exp06_wait.current_contribution.sum())},{'experiment':'canonical','year_scope':'all','decision':'FLEXIBLE','N':len(canonical_flex),'formula':'spot - historical FLEX','economic_contribution':float((canonical_flex.spot_cost-canonical_flex.historical_flex_cost).sum())},{'experiment':'EXP-06','year_scope':'all','decision':'NON_WAIT','N':len(exp06_nonwait),'formula':'spot - historical FLEX','economic_contribution':float((exp06_nonwait.spot_cost-exp06_nonwait.historical_flex_cost).sum())}]); display(ledger)
OUT=REPO/'outputs'/'forensic_final'; OUT.mkdir(parents=True, exist_ok=True); audit.to_csv(OUT/'row_level_audit.csv', index=False); ledger.to_csv(OUT/'population_ledger.csv', index=False); decomp.to_csv(OUT/'yearly_vessel_decomposition.csv', index=False); threshold_rows.to_csv(OUT/'threshold_audit.csv', index=False); flex_ledger.to_csv(OUT/'flex_reconciliation.csv', index=False); final_reconciliation.to_csv(OUT/'final_reconciliation.csv', index=False)
print('Generated fresh forensic outputs:', OUT)